# Advanced Features Exploration

Template for measuring incremental value from LLM news and on-chain feature groups. Keep all joins time-series safe: only use signal partitions and labels available at each `as_of` timestamp.

In [ ]:
from pathlib import Path

import pandas as pd

SIGNALS = Path("../data/processed/signals")
frames = [pd.read_parquet(path) for path in SIGNALS.glob("date=*/signals.parquet")]
signals = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
signals.head()

In [ ]:
if not signals.empty and "features" in signals:
    feature_frame = pd.json_normalize(signals["features"])
    analysis = pd.concat([signals.drop(columns=["features"]), feature_frame], axis=1)
else:
    analysis = signals

advanced_cols = [
    col
    for col in analysis.columns
    if col.startswith(("llm_", "onchain_", "news_velocity_", "cross_source_"))
]
analysis[advanced_cols].describe() if advanced_cols else "No advanced feature columns found."

## Incremental Value Checklist

- Compare Phase 1 features vs Phase 1 + advanced features on identical time splits.
- Track Brier, log loss, calibration slope/intercept, and threshold hit rates.
- Compare cost per scored market against edge improvement.
- Audit cached prompts/responses in `data/processed/llm_cache` for failure modes.